# Multimodal Retrieval Augmented Generation (RAG) with Gemini, Vertex AI Vector Search, and LangChain

---

# Part 2: Process User Query

In [ ]:
%%html
<marquee style='width: 100%;'><h1 style='color: Gray; letter-spacing: 12.5px;'><b style='color: Black;'>SKY</b>VAR</h1>Rise up with the unlimited sky</marquee>

## Getting Started

### Install Vertex AI SDK for Python and other dependencies

In [ ]:
%pip install -U -q google-cloud-aiplatform langchain-core langchain-google-vertexai langchain-text-splitters langchain-community "unstructured[all-docs]" pypdf pydantic lxml pillow matplotlib opencv-python tiktoken

### Restart current runtime

To use the newly installed packages in this Jupyter runtime, you must restart the runtime. You can do this by running the cell below, which will restart the current kernel.

In [ ]:
# Restart kernel after installs so that your environment can access the new packages
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

<div class="alert alert-block alert-warning">
<b>⚠️ The kernel is going to restart. Please wait until it is finished before continuing to the next step. ⚠️</b>
</div>


### Authenticate your notebook environment (Colab only)

If you are running this notebook on Google Colab, run the following cell to authenticate your environment. This step is not required if you are using [Vertex AI Workbench](https://cloud.google.com/vertex-ai-workbench).

In [ ]:
import sys

# Additional authentication is required for Google Colab
if "google.colab" in sys.modules:
    # Authenticate user to Google Cloud
    from google.colab import auth

    auth.authenticate_user()

### Define Google Cloud project information

In [ ]:
PROJECT_ID = "XXXXX"  # @param {type:"string"}
LOCATION = "XXXXX"  # @param {type:"string"}

# For Vector Search Staging
GCS_BUCKET = "XXXXX"  # @param {type:"string"}
CORPUS_FOLDER = "corpus"  # @param {type:"string"}
GCS_BUCKET_URI = f"gs://{GCS_BUCKET}"

### Initialize the Vertex AI SDK

In [ ]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=GCS_BUCKET_URI)

### Import libraries

In [ ]:
import ast
import base64
import io
import logging
import os
import re

from PIL import Image
from IPython.display import Markdown, display
from google.cloud import storage
from langchain.chains import create_history_aware_retriever
from langchain.retrievers import EnsembleRetriever, MultiQueryRetriever
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_google_vertexai import (
    ChatVertexAI,
    VectorSearchVectorStore,
    VertexAI,
    VertexAIEmbeddings,
)
from langchain_google_vertexai.vectorstores.document_storage import GCSDocumentStorage
from pathlib import Path
from pydantic import BaseModel
from urllib.parse import quote

### Define model information

- [Vertex AI - Model Information](https://cloud.google.com/vertex-ai/generative-ai/docs/learn/models)

In [ ]:
MODEL_NAME = "gemini-2.0-flash-001"
GEMINI_OUTPUT_TOKEN_LIMIT = 8192

EMBEDDING_MODEL_NAME = "text-embedding-004"
EMBEDDING_TOKEN_LIMIT = 2048

TOKEN_LIMIT = min(GEMINI_OUTPUT_TOKEN_LIMIT, EMBEDDING_TOKEN_LIMIT)

### Define index information

Connect to exist vertex AI vector search:

*   Copy index id from https://console.cloud.google.com/vertex-ai/matching-engine/indexes
*   Copy endpoint id from https://console.cloud.google.com/vertex-ai/matching-engine/index-endpoints



In [ ]:
INDEX_ID = 'XXXXXXXXXXXXXXXXXXXX' #@param {type: "string"}
ENDPOINT_ID = 'XXXXXXXXXXXXXXXXXXXX' #@param {type: "string"}

## Create retrievers

- Create [`VectorSearchVectorStore`](https://api.python.langchain.com/en/latest/vectorstores/langchain_google_vertexai.vectorstores.vectorstores.VectorSearchVectorStore.html) with Vector Search Index ID and Endpoint ID.
- Use [`textembedding-gecko`](https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/text-embeddings) as embedding model.

In [ ]:
# @title vectorstore
# The vectorstore to use to index the summaries
vectorstore = VectorSearchVectorStore.from_components(
    project_id=PROJECT_ID,
    region=LOCATION,
    gcs_bucket_name=GCS_BUCKET,
    index_id=INDEX_ID,
    endpoint_id=ENDPOINT_ID,
    embedding=VertexAIEmbeddings(model_name=EMBEDDING_MODEL_NAME),
    stream_update=True,
)

- Create Multi-Vector Retriever using the vector store you created.
- Since vector stores only contain the embedding and an ID, you'll also need to create a document store indexed by ID to get the original source documents after searching for embeddings.

In [ ]:
# @title docstore
try:
    storage_client = storage.Client()
    bucket = storage_client.bucket(GCS_BUCKET)
    if not bucket.exists():
        raise ValueError(f"Bucket '{GCS_BUCKET}' does not exist.")
except Exception as e:
    print(f"An error occurred: {e}")

docstore = GCSDocumentStorage(bucket, "chunks")

In [ ]:
# @title retriever_multi_vector_img
id_key = "doc_id"
retriever_multi_vector_img = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    id_key=id_key,
    search_kwargs={"k":24}
)

In [ ]:
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

In [ ]:
# @title multi_query_retriever
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=retriever_multi_vector_img,
    llm=VertexAI(
        temperature=0,
        model_name=MODEL_NAME,
        max_output_tokens=TOKEN_LIMIT
    )
)

In [ ]:
# @title ensemble_retriever
ensemble_retriever = EnsembleRetriever(
    retrievers=[retriever_multi_vector_img,
                multi_query_retriever],
    weights=[0.8, 0.2]
)

In [ ]:
# @title history_aware_retriever
history_prompt_str = """Chat history data and the user's last question \
which may relate to the context in the chat history, formulate a stand-alone question \
that can be understood without the chat history. Do not answer the question, \
just rephrase it if necessary, otherwise return the last question as is without adding a personal opinion."""

history_prompt = ChatPromptTemplate.from_messages(
    [
        MessagesPlaceholder(variable_name="chat_history"),
        ("user", "{input}"),
        ("user", history_prompt_str)
    ]
)

history_aware_retriever = create_history_aware_retriever(
    llm=VertexAI(
        temperature=0,
        model_name=MODEL_NAME,
        max_output_tokens=TOKEN_LIMIT
    ),
    retriever=ensemble_retriever,
    prompt=history_prompt
)

## Create Chain with Retriever and Gemini LLM

In [ ]:
def is_hebrew(text):
  return any("\u0590" <= letter <= "\u05EA" for letter in text)

In [ ]:
translation_prompt = """
    You are a professional technical translator for helicopter systems documentation.
    Your job is to translate the following Hebrew question to **natural and accurate English** that fits the terminology and phrasing style used in official helicopter manuals.

    **Important Guidelines:**
    1. Use the dictionary below only as a **reference for technical terms** — do not blindly follow it if a more natural or appropriate term exists based on context.
    2. If the Hebrew term has a plural, gender, or tense adjustment needed, apply it naturally in English.
    3. If the question contains **acronyms written in English** (such as RFM, EEC, NR), leave them **unchanged** exactly as written.
       - However, **adjust capitalization as needed** to match official helicopter documentation style (e.g., "rpm" → "RPM", "eec" → "EEC" if referring to a system).
    4. If the question contains acronyms written in Hebrew, translate them to the most appropriate technical term used in helicopter documentation, using the provided dictionary if relevant.
    5. Ensure **context-aware terminology matching**, adapting word forms as needed:
       - **Pluralization** (e.g., "ממסר" → "Transmission", "ממסרים" → "Transmissions").
       - **Gender and verb tense adjustments** for natural readability in English.
    6. **Strictly maintain any values, acronyms, or numbers as they appear** in the original text—do not modify or translate numerical values, units, or predefined English abbreviations.
    7. Do not add explanations, assumptions, or background information—**translate only the question itself**.

    Please return only the translated question, nothing else.
    """


In [ ]:
# @title Dictionary
dictionary = {
    "ממסר אביזרים": "accessory gearbox",
    "להתאים": "adjust",
    "אחורי": "aft",
    "להחריף": "aggravate",
    "מדחס אוויר": "Air Compressor",
    "בערך": "approximately",
    "מערכת אזהרה קולית": "Audio Warning Generator (AWG)",
    "טייס אוטומטי": "Autopilot (Helipilot)",
    "להמנע": "avoid",
    "ציר": "axis",
    "זוויות הטייה": "Bank angles",
    "מצבר/סוללה": "Battery",
    "מתחת": "below",
    "משאבת הגבר": "Boost Pump",
    "פס צבירה": "Bus bar",
    "תא נוסעים": "cabin",
    "נורת אזהרה": "Caution Light",
    "חופת תא טייס": "Cockpit Canopy",
    "מדחס": "Compressor",
    "תנאים": "conditions",
    "לאשר": "confirm",
    "לשכך": "cushion",
    "מנותק/מכובה": "deactivated",
    "להוריד": "decrease",
    "להגדיר": "determined",
    "בקרת מנוע אלקטרונית": "Electronic Engine Control (EEC)",
    "אזורים סגורים": "enclosed areas",
    "מנוע": "Engine",
    "מצערת מנוע": "Engine Twist Grip",
    "להימלט": "evacuate",
    "לחרוג": "exceed",
    "מטף כיבוי אש": "fire extinguisher",
    "מונעי הנפנוף": "flap restraint mechanism",
    "קדימה": "forward",
    "מד כמות דלק": "Fuel Quantity Indicator",
    "גנרטור": "Generator",
    "תקלת גנרטור": "Generator Failure",
    "מרחק מרבי": "glide distance",
    "משאבת דלק לחץ גבוה": "HP Fuel Pump",
    "מערכת הידראולית": "Hydraulic System",
    "לעלות": "increase",
    "חיווי INV OFF": "INV OFF Caution Message",
    "מהפך": "Inverter",
    "לבודד": "isolate",
    "כני נסע": "Landing Gear",
    "ידית": "lever",
    "ממסר ראשי": "Main Gearbox (MGB)",
    "חיישן טמפרטורת שמן ממסר ראשי": "Main Gearbox Oil Temperature Sensor",
    "משאבת הידראוליקה ראשית": "Main Hydraulic Pump",
    "לתחזק": "maintain",
    "תמרון": "manoeuver",
    "בקרת מנוע מכנית": "Mechanical Engine Control (MEC)",
    "לבקר": "monitor",
    'נפילת סל"ד': "NR decay",
    'מקזז סל"ד': "NR trim",
    "תאורת ז'ורנאל (מותאמת NVG)": "NVIS Compatible Lighting",
    "לצפות": "observe",
    "להשיג": "obtain",
    "נוסעים": "occupants",
    "תנודה": "oscillation",
    "מקביל": "parallel",
    "חימום צינורות פיטו": "Pitot Tube Heating",
    "טורבינת כוח": "Power Turbine (N2)",
    "כמות": "quantity",
    "מד גובה אלקטרוני": "radar altimeter",
    "שיעור הנמכה": "rate of descent",
    "להפחית": "reduce",
    "תפקיד": "role",
    "תפקידים": "roles",
    "חיישן RPM": "RPM Sensor",
    "חלוניות שמן ממסר": "sight level gauge/ sight glass / sight gauge",
    "מגלש": "skid",
    "מגלשיים": "Skids / Landing skids",
    "מתייצב": "stabilized",
    "מספק": "supply",
    "מערכת": "system",
    "מגבלת טמפרטורה": "Temperature Limit",
    "נטייה": "tendency",
    "מצערת": "throttle",
    "ממסר": "Transmission",
    "תיבת ממסר": "Transmission",
    "שמן ממסר": "Transmission oil",
    "מקזז": "trim",
    "שיעור הפנייה": "turn rate",
    "שמיש": "usable",
    "פתח אוורור": "vent",
    "אוורור": "ventilation",
    "נורת התראה": "Warning Light",
    "סבסוב": "yaw",
}

In [ ]:
def query_processing(conversation_history):
    query = conversation_history["messages"][-1]["content"]
    if not is_hebrew(query):
        return conversation_history

    prompt = f"{translation_prompt}\n\nHebrew Question:\n{query}\n\nReference Dictionary (for technical terms only):\n{dictionary}"

    model = ChatVertexAI(
        model_name=MODEL_NAME,
        max_output_tokens=TOKEN_LIMIT,
        temperature=0.2,
    )
    result = model.invoke(prompt)

    conversation_history["messages"][-1]["content"] = result.content.strip()
    return conversation_history

In [ ]:
def is_base64(s):
    try:
        return base64.b64encode(base64.b64decode(s)) == s.encode()
    except Exception:
        return False


def split_image_text_types(docs):
    b64_images = []
    texts = []
    for doc in docs:
        metadata = doc.metadata
        doc.metadata = {"doc_id": metadata['doc_id']}
        if is_base64(doc.page_content):
            print(doc.page_content[:10])
            b64_images.append(doc)
        else:
            texts.append(doc)
    return {"images": b64_images, "texts": texts}

In [ ]:
def sources_retrieval(conversation):
    chat_history = conversation["messages"]
    query = conversation["messages"][-1]["content"]
    history_docs = history_aware_retriever.invoke(
        {"input": query, "chat_history": chat_history}
    )#[:30]
    # Sisters' code here
    source_docs = split_image_text_types(history_docs)
    input_data = {"context": source_docs, "question": query, "history": chat_history}
    return input_data

In [ ]:
model_instructions = """ You are a learning assistant tasked with helping trainees
in the pilot course understand the 'Ofer' helicopter systems and operating instructions.
Your primary goal is to provide **technically accurate, clear, and detailed answers**
that strictly align with the official helicopter documentation and operational guidelines.
You will answer questions based on the full context of the conversation history, ensuring accuracy and relevance.
"""

In [ ]:
def format_model_input(data_dict):
    formatted_chunks = str(data_dict["context"]["texts"])
    full_prompt = f"User-provided question: {data_dict['question']}\n conversation history:{data_dict['history']}\nText and / or tables:\n{formatted_chunks}"
    messages = [
        {
            "type": "text",
            "text": (full_prompt),
        }
    ]
    if data_dict["context"]["images"]:
        for image in data_dict["context"]["images"]:
            messages.append(
                {
                    "type": "text",
                    "text": f"metadata:\n{image.metadata}"
                }
            )
            messages.append(
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{image.page_content}"},
                }
            )
    return [HumanMessage(content=messages)]

In [ ]:
response_schema = {
    "type": "object",
    "properties": {
        "markdown_answer_with_reasoning": {
            "type": "string",
            "description": """Provide a **concise, accurate, and structured response** based strictly on the official documentation of the 'Ofer' helicopter.
- **Answer the specific question directly**, focusing only on essential operational details (e.g., numerical thresholds, required conditions, or critical steps).
- If multiple configurations, modes, or conditions exist, include only those relevant to the question.
- **Use official terminology** and adhere strictly to documented operational logic.
- If exact numbers, steps, or limitations exist, **provide them exactly as documented**.
- If the documentation does **not** fully answer the question, state that clarification is required rather than making assumptions.
**Important rules:**
- :exclamation:️Never include links to images or documents inside the answer itself.
- :exclamation:️Do not reuse, hallucinate, or fabricate image URLs or embedded media.
- :exclamation:️Do not mention or reference any document/image IDs in the answer text.
- :white_check_mark: All source references (text or image) must appear **only** in the `doc_ids` field.
### **Handling Image Relevance**
Include an image in the `doc_ids` field only if it meets **one of the following conditions**:
1. It was **explicitly requested** by the user.
2. It was **used directly** to formulate part of the answer.
3. It provides **essential visual enrichment** that helps clarify or support the textual answer (e.g., a diagram of a specific component mentioned in the answer).
Avoid including images that are merely thematically related or part of generic startup/checklist procedures unless they are directly connected to the specific question and answer.
"""
        },
        "doc_ids": {
            "type": "array",
            "items": {"type": "string"},
            "description": """Return the list of documents and images that were essential for answering the question.
**Text sources:**
- Include only text documents that were **actually used to formulate the answer**.
- If the answer relies on information, logic, or procedures, it must include the corresponding text source.
- :white_check_mark: In most cases, there should be **at least one relevant text source** in this list.
**Images and diagrams:**
- Include images only if:
  - They were directly used in the answer,
  - Were explicitly requested by the user,
  - Or clearly enrich the understanding of the answer in a visual way.
- If multiple similar images exist, include only the most representative (up to 3 total).
- :exclamation:️Do not include an image if the answer would be equally valid without it.
Only include raw `doc_id` strings. Do not embed links, captions, or explanations.
"""
        }
    },
    "required": ["markdown_answer_with_reasoning", "doc_ids"],
}

In [ ]:
SIGN_SERVER_URL = 'https://sign_server_url'  # @param {type:"string"}
IMAGES_SIGN_URL = "get_image"
LINKS_SIGN_URL = "get_link"
REQUEST_PARAM = "?url="

In [ ]:
from IPython import display

In [ ]:
def set_links(result):
    sources_links = get_sources(result['doc_ids'])
    return {'answer': result['markdown_answer_with_reasoning'],
            **sources_links
            }


def get_sources(doc_ids):
    unique_doc_ids = set(doc_ids)
    chunks = get_chunks(unique_doc_ids)
    result = {"images": [], "links": []}

    {
        (
            result["images"].append(get_image_path(chunk))
            if is_base64(chunk.page_content)
            else result["links"].append(get_link(chunk))
        )
        for chunk in chunks
    }

    return result


def get_chunks(docs_ids):
    return ensemble_retriever.retrievers[0].docstore.mget(docs_ids)


def get_image_path(image_chunk):
    sign_server_url = os.path.join(SIGN_SERVER_URL,
                                   IMAGES_SIGN_URL)
    link = image_chunk.metadata['url']
    display.display(display.Image(base64.b64decode(image_chunk.page_content)))
    return f"![]({sign_server_url}{REQUEST_PARAM}{link})".replace('\\', '/')


def get_link(chunk):
    chunk_metadata = get_chunk_metadata(chunk)
    return f"[{get_link_preview(chunk_metadata)}]({get_document_link(chunk_metadata)})"


def get_chunk_metadata(chunk):
    print(chunk.metadata)
    return [chunk.metadata['filename'], chunk.metadata['page_number']]


def get_link_preview(chunk_metadata):
    return f"{chunk_metadata[0]} P. {chunk_metadata[1]}"


def get_document_link(chunk_metadata):
    filename = quote(chunk_metadata[0])
    sign_server_url = os.path.join(SIGN_SERVER_URL,
                                   LINKS_SIGN_URL)
    link = os.path.join(GCS_BUCKET,
                        CORPUS_FOLDER,
                        get_filename_without_prefix(filename),
                        f"{filename}&page={chunk_metadata[1]}"
                        )
    return f"{sign_server_url}{REQUEST_PARAM}{link}".replace('\\', '/')


def get_filename_without_prefix(filename):
    return Path(filename).stem

In [ ]:
chain_multimodal_rag = (
    RunnableLambda(query_processing)
    | RunnableLambda(sources_retrieval)
    | RunnableLambda(format_model_input)
    | ChatVertexAI(
        temperature=0,
        top_p = 0.1,
        model_name=MODEL_NAME,
        max_output_tokens=TOKEN_LIMIT,
        response_mime_type="application/json",
        response_schema=response_schema,
        system_instruction=model_instructions,
    )
    # | model
    | JsonOutputParser()
    | RunnableLambda(set_links)
)

## Get Answer

In [ ]:
query = "Your query here" #@param {type: "string"}
messages = [{'role': 'user', 'content': query}]

In [ ]:
history = {
    'stream': True,
    'model': 'Chayapipeline',
    'messages': messages,
    'metadata': {
        'user_id': '60024248-99a6-4784-8c02-aeb09a871811',
        'chat_id': 'b3faf3f0-a792-4310-afa3-ebce2448b1cc',
        'message_id': '9215e2ac-f93c-4c5e-a6c9-23837b269c6a',
        'session_id': 'gHYKomRI7kq1xwx7AAAR',
        'tool_ids': None,
        'files': None,
        'features': {'web_search': False}
        }
    }

In [ ]:
result = chain_multimodal_rag.invoke(history)
Markdown(result['answer'])

In [ ]:
result

In [ ]:
for link in result['links']:
    print(link)